<a href="https://colab.research.google.com/github/jjmin2/CV_Project_9team/blob/main/CV_Evaluation_Comparison_of_AUROC_and_IoU.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import sys

sys.path.append('/content/drive/MyDrive/CV/activeLearning')
sys.path.append('/content/drive/MyDrive/CV/activeLearning/sobel_detector.py')
sys.path.append('/content/drive/MyDrive/CV/activeLearning/padim.py')

In [4]:
import os
import glob
import torch
import numpy as np
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms

# 1. 데이터셋 클래스 선언
class MVTecDataset(Dataset):
    def __init__(self, data_dir, category, split='train'):
        self.data_dir = os.path.join(data_dir, category, split)
        self.split = split
        self.image_paths = sorted(glob.glob(os.path.join(self.data_dir, '**', '*.png'), recursive=True))

        self.transform = transforms.Compose([
            transforms.Resize((256, 256)),
            transforms.ToTensor()
        ])
        self.mask_transform = transforms.Compose([
            transforms.Resize((256, 256), interpolation=transforms.InterpolationMode.NEAREST),
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        img_tensor = self.transform(image)

        if self.split == 'train':
            label = 0
            mask_tensor = torch.zeros((1, 256, 256))
        else:
            defect_type = os.path.basename(os.path.dirname(img_path))
            if defect_type == 'good':
                label = 0
                mask_tensor = torch.zeros((1, 256, 256))
            else:
                label = 1
                mask_path = img_path.replace('test', 'ground_truth').replace('.png', '_mask.png')
                if os.path.exists(mask_path):
                    mask_tensor = self.mask_transform(Image.open(mask_path).convert('L'))
                else:
                    mask_tensor = torch.zeros((1, 256, 256))

        return img_tensor, torch.tensor(label, dtype=torch.float32), mask_tensor, img_path

DATA_DIR = '/content/drive/MyDrive/CV/activeLearning/mvtec_anomaly_detection'
CATEGORY = 'bottle'

dataset_train = MVTecDataset(DATA_DIR, CATEGORY, split='train')
dataset_val = MVTecDataset(DATA_DIR, CATEGORY, split='val')
dataset_test = MVTecDataset(DATA_DIR, CATEGORY, split='test')

In [6]:
import os
import glob
import json
import time
import numpy as np
import pandas as pd
import cv2
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.metrics import roc_auc_score

# 팀원 모듈 임포트
from sobel_detector import SobelDetector
from padim import PaDiMDetector

# ==========================================
# 0. 평가지표 및 헬퍼 함수
# ==========================================
def calculate_auroc(y_true, y_scores): return roc_auc_score(y_true, y_scores)
def calculate_pixel_auroc(mask_true, score_map): return roc_auc_score(mask_true.flatten(), score_map.flatten())
def calculate_iou(mask_true, mask_pred):
    intersection, union = np.logical_and(mask_true, mask_pred).sum(), np.logical_or(mask_true, mask_pred).sum()
    return np.nan if union == 0 else intersection / union
def tensor_to_bgr(tensor):
    img_rgb = (tensor.squeeze(0).permute(1, 2, 0).numpy() * 255).astype(np.uint8)
    return cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)

# ==========================================
# 1. 데이터셋 클래스
# ==========================================
class MVTecDataset(Dataset):
    def __init__(self, data_dir, category, split='train', split_json=None):
        self.split = split
        self.transform = transforms.Compose([transforms.Resize((256, 256)), transforms.ToTensor()])
        self.mask_transform = transforms.Compose([transforms.Resize((256, 256), interpolation=transforms.InterpolationMode.NEAREST), transforms.ToTensor()])

        self.image_paths = []

        if split in ['train', 'val']:
            if split_json and os.path.exists(split_json):
                with open(split_json, 'r') as f:
                    splits = json.load(f)

                # 웹앱 JSON 기준 키 매핑
                json_key = 'statistics_images' if split == 'train' else 'validation_images'
                rel_paths = splits.get(json_key, [])

                # JSON에 명시된 경로를 절대 경로로 바로 연결
                for rel_p in rel_paths:
                    full_path = os.path.join(data_dir, category, rel_p)
                    self.image_paths.append(full_path)
            else:
                # JSON 파일이 없으면 기존처럼 전체 폴더 로드
                base_path = os.path.join(data_dir, category, 'train')
                self.image_paths = sorted(glob.glob(os.path.join(base_path, '**', '*.png'), recursive=True))

        elif split == 'test':
            base_path = os.path.join(data_dir, category, 'test')
            self.image_paths = sorted(glob.glob(os.path.join(base_path, '**', '*.png'), recursive=True))

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        img_tensor = self.transform(Image.open(img_path).convert('RGB'))

        if self.split in ['train', 'val']:
            label, mask_tensor = 0, torch.zeros((1, 256, 256))
        else:
            defect_type = os.path.basename(os.path.dirname(img_path))
            if defect_type == 'good':
                label, mask_tensor = 0, torch.zeros((1, 256, 256))
            else:
                label = 1
                mask_path = img_path.replace('test', 'ground_truth').replace('.png', '_mask.png')
                if os.path.exists(mask_path):
                    mask_tensor = self.mask_transform(Image.open(mask_path).convert('L'))
                else:
                    mask_tensor = torch.zeros((1, 256, 256))
        return img_tensor, torch.tensor(label, dtype=torch.float32), mask_tensor, img_path

# ==========================================
# 2. 경로 설정 및 모델 세팅
# ==========================================
print("1. 데이터셋 및 모델 세팅 시작...")
DATA_DIR = '/content/drive/MyDrive/CV/activeLearning/mvtec_anomaly_detection'
JSON_PATH = '/content/drive/MyDrive/CV/activeLearning/bottle.json'
CATEGORY = 'bottle'

dataset_train = MVTecDataset(DATA_DIR, CATEGORY, split='train', split_json=JSON_PATH)
dataset_val = MVTecDataset(DATA_DIR, CATEGORY, split='val', split_json=JSON_PATH)
dataset_test = MVTecDataset(DATA_DIR, CATEGORY, split='test')

print(f"- 학습(train): {len(dataset_train)}장, 검증(val): {len(dataset_val)}장 로드 완료 (웹앱과 동일)")

# --- Sobel 세팅 ---
sobel_model = SobelDetector()
train_images = [dataset_train[i][0] for i in range(len(dataset_train))]
val_images = [dataset_val[i][0] for i in range(len(dataset_val))]
sobel_model.fit(train_images)
sobel_model.calibrate(val_images)
print("- [Sobel] 정상 통계 추정 및 임계값 보정 완료")

# --- PaDiM 세팅 ---
padim_model = PaDiMDetector()
train_pt_path = '/content/drive/MyDrive/bottle_train_features.pt'
padim_model.fit_from_file(train_pt_path)

val_score_maps_padim = [padim_model.score_map(tensor_to_bgr(img)) for img in val_images]
padim_pixel_threshold = np.percentile(np.stack(val_score_maps_padim), 99.5)
print(f"- [PaDiM] 정상 통계 로드 및 픽셀 임계값({padim_pixel_threshold:.4f}) 보정 완료\n")

# ==========================================
# 3. 테스트 데이터 통합 평가 진행
# ==========================================
print("2. 테스트 데이터 통합 평가 진행 중...")
results = {
    'Sobel': {'y_true': [], 'y_scores': [], 'px_auroc': [], 'iou': [], 'time': []},
    'PaDiM': {'y_true': [], 'y_scores': [], 'px_auroc': [], 'iou': [], 'time': []}
}

for img_tensor, label, mask_tensor, _ in DataLoader(dataset_test, batch_size=1, shuffle=False):
    single_img, img_bgr, mask_numpy, lbl = img_tensor.squeeze(0), tensor_to_bgr(img_tensor), mask_tensor.squeeze().numpy(), label.item()
    results['Sobel']['y_true'].append(lbl)
    results['PaDiM']['y_true'].append(lbl)

    # Sobel
    sobel_res = sobel_model.predict(single_img)
    results['Sobel']['time'].append(sobel_res["elapsed_ms"])
    results['Sobel']['y_scores'].append(sobel_res["image_score"])

    # PaDiM
    start_time = time.perf_counter()
    padim_score_map = padim_model.score_map(img_bgr)
    results['PaDiM']['time'].append((time.perf_counter() - start_time) * 1000)
    results['PaDiM']['y_scores'].append(float(np.max(padim_score_map)))

    # 픽셀 평가 (불량일 경우 마스크 비교)
    if len(np.unique(mask_numpy)) > 1:
        results['Sobel']['px_auroc'].append(calculate_pixel_auroc(mask_numpy, sobel_res["score_map"]))
        results['PaDiM']['px_auroc'].append(calculate_pixel_auroc(mask_numpy, padim_score_map))
    if lbl == 1:
        sobel_iou = calculate_iou(mask_numpy, sobel_res["mask"])
        if not np.isnan(sobel_iou): results['Sobel']['iou'].append(sobel_iou)

        padim_pred_mask = (padim_score_map > padim_pixel_threshold).astype(np.uint8)
        padim_iou = calculate_iou(mask_numpy, padim_pred_mask)
        if not np.isnan(padim_iou): results['PaDiM']['iou'].append(padim_iou)

# ==========================================
# 4. 최종 지표 산출 및 통합 CSV 저장
# ==========================================
final_data = []
print("-" * 50)
for method in ['Sobel', 'PaDiM']:
    r = results[method]
    img_auroc, px_auroc = calculate_auroc(r['y_true'], r['y_scores']), np.mean(r['px_auroc']) if r['px_auroc'] else 0
    mean_iou, mean_time = np.mean(r['iou']) if r['iou'] else 0, np.mean(r['time'])
    print(f"[{method}] Image AUROC: {img_auroc:.4f} | Pixel AUROC: {px_auroc:.4f} | Mean IoU: {mean_iou:.4f} | Time: {mean_time:.2f} ms")
    final_data.append({'Method': method, 'Image AUROC': img_auroc, 'Pixel AUROC (Mean)': px_auroc, 'Mean IoU (Defective only)': mean_iou, 'Processing Time Mean (ms)': mean_time})

df_combined = pd.DataFrame(final_data)
df_combined.to_csv('/content/drive/MyDrive/Evaluation_Results_WebAligned.csv', index=False)
print("\n 결과가 구글 드라이브(Evaluation_Results_WebAligned.csv)에 저장되었습니다")

1. 데이터셋 및 모델 세팅 시작...
- 학습(train): 150장, 검증(val): 38장 로드 완료 (웹앱과 동일)
- [Sobel] 정상 통계 추정 및 임계값 보정 완료
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 143MB/s]


- [PaDiM] 정상 통계 로드 및 픽셀 임계값(8.2286) 보정 완료

2. 테스트 데이터 통합 평가 진행 중...
--------------------------------------------------
[Sobel] Image AUROC: 0.9817 | Pixel AUROC: 0.7951 | Mean IoU: 0.2019 | Time: 4.36 ms
[PaDiM] Image AUROC: 0.9873 | Pixel AUROC: 0.9818 | Mean IoU: 0.3380 | Time: 138.59 ms

 결과가 구글 드라이브(Evaluation_Results_WebAligned.csv)에 저장되었습니다


In [7]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

# 1. 시각화 결과를 저장할 구글 드라이브 폴더 생성
save_dir = '/content/drive/MyDrive/CV/activeLearning/PPT_Visualizations'
os.makedirs(save_dir, exist_ok=True)

# 2. 히트맵 오버레이 함수 (웹앱 화면과 동일한 시각적 효과)
def draw_heatmap_overlay(img_rgb, score_map):
    # 점수 맵을 0~255로 정규화
    norm_map = cv2.normalize(score_map, None, 0, 255, cv2.NORM_MINMAX, dtype=cv2.CV_8U)
    # Jet 컬러맵 적용
    heatmap = cv2.applyColorMap(norm_map, cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    # 원본 이미지와 5:5 비율로 반투명하게 겹치기
    overlay = cv2.addWeighted(img_rgb, 0.5, heatmap, 0.5, 0)
    return overlay

print("PPT용 시각화 이미지 추출 시작...")

# 3. 결함 유형(good, broken_large, broken_small, contamination)별로 샘플 추출
sampled_types = {}
max_samples_per_type = 2  # 각 유형별로 추출할 이미지 수

for i in range(len(dataset_test)):
    img_tensor, _, mask_tensor, img_path = dataset_test[i]
    defect_type = os.path.basename(os.path.dirname(img_path))

    # 이미 목표 수량을 채운 유형은 건너뜀
    if sampled_types.get(defect_type, 0) >= max_samples_per_type:
        continue

    sampled_types[defect_type] = sampled_types.get(defect_type, 0) + 1

    # 데이터 변환
    single_img = img_tensor.squeeze(0)
    img_bgr = tensor_to_bgr(img_tensor)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    mask_numpy = mask_tensor.squeeze().numpy()

    # 모델 예측
    sobel_score_map = sobel_model.predict(single_img)["score_map"]
    padim_score_map = padim_model.score_map(img_bgr)

    # 히트맵 생성
    padim_overlay = draw_heatmap_overlay(img_rgb, padim_score_map)
    sobel_overlay = draw_heatmap_overlay(img_rgb, sobel_score_map)

    # 4. 가로 배치 시각화 (1행 4열)
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))

    axes[0].imshow(img_rgb)
    axes[0].set_title(f'Original ({defect_type})', fontsize=16, fontweight='bold', pad=10)
    axes[0].axis('off')

    axes[1].imshow(mask_numpy, cmap='gray')
    axes[1].set_title('Ground Truth Mask', fontsize=16, fontweight='bold', pad=10)
    axes[1].axis('off')

    axes[2].imshow(padim_overlay)
    axes[2].set_title('PaDiM Heatmap', fontsize=16, fontweight='bold', pad=10)
    axes[2].axis('off')

    axes[3].imshow(sobel_overlay)
    axes[3].set_title('Sobel Heatmap', fontsize=16, fontweight='bold', pad=10)
    axes[3].axis('off')

    plt.tight_layout()

    # 5. 고화질 파일로 저장
    filename = f"Compare_{defect_type}_{os.path.basename(img_path)}"
    save_path = os.path.join(save_dir, filename)
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close() # 메모리 확보를 위해 출력 창 닫기

    print(f"[{defect_type}] 저장 완료: {filename}")

    # 모든 유형을 2장씩 모았다면 반복문 종료
    if sum(sampled_types.values()) >= 8:
        break

print(f"\n완료. 구글 드라이브의 '{save_dir}' 경로를 확인해 보세요.")

PPT용 시각화 이미지 추출 시작...
[broken_large] 저장 완료: Compare_broken_large_000.png
[broken_large] 저장 완료: Compare_broken_large_001.png
[broken_small] 저장 완료: Compare_broken_small_000.png
[broken_small] 저장 완료: Compare_broken_small_001.png
[contamination] 저장 완료: Compare_contamination_000.png
[contamination] 저장 완료: Compare_contamination_001.png
[good] 저장 완료: Compare_good_000.png
[good] 저장 완료: Compare_good_001.png

완료. 구글 드라이브의 '/content/drive/MyDrive/CV/activeLearning/PPT_Visualizations' 경로를 확인해 보세요.
